# Driver lifetime value

Lyft / StrataScratch. Three tables: who onboarded, completed rides, event log.

What is a driver worth over the time they stay, and are we averaging two different kinds of people?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

kaggle = Path('/kaggle/input/datasets/paprikaryash/datasets1i')
DATA = kaggle if kaggle.exists() else Path('data')

drivers = pd.read_csv(DATA / 'driver_ids.csv')
rides = pd.read_csv(DATA / 'ride_ids.csv')
ts = pd.read_csv(DATA / 'ride_timestamps.csv')
print(drivers.shape, rides.shape, ts.shape)
print(list(drivers.columns))
print(list(rides.columns))
print(list(ts.columns))


## First look

Distance is meters, duration is seconds, prime time is an integer percent.


In [ ]:
display(drivers.head())
display(rides.head())
display(ts.head(10))
print(drivers.isna().sum())
print(rides.isna().sum())
print(ts.isna().sum())
print('unique drivers', drivers.driver_id.nunique())
print('unique rides', rides.ride_id.nunique())
print('event ids', ts.ride_id.nunique(), 'in', len(ts), 'rows')


In [ ]:
drivers['driver_onboard_date'] = pd.to_datetime(drivers['driver_onboard_date'])
ts['timestamp'] = pd.to_datetime(ts['timestamp'])
print('onboard', drivers.driver_onboard_date.min(), '->', drivers.driver_onboard_date.max())
print('events', ts.timestamp.min(), '->', ts.timestamp.max())
print('drivers with no ride row', drivers.driver_id.nunique() - rides.driver_id.nunique())
print('rides missing from ts', rides.ride_id.nunique() - ts.ride_id.nunique())
print('ts ids not in rides', ts.ride_id.nunique() - rides.ride_id.nunique())


## Fare

Rate card. Fee comes off the top so pay cannot go negative.


In [ ]:
BASE, PER_MILE, PER_MIN = 2.00, 1.15, 0.22
FEE, MIN_FARE, MAX_FARE = 1.75, 5.00, 400.00

rides = rides.copy()
rides['miles'] = rides['ride_distance'] / 1609.34
rides['minutes'] = rides['ride_duration'] / 60
raw = BASE + PER_MILE * rides['miles'] + PER_MIN * rides['minutes']
raw = raw * (1 + rides['ride_prime_time'] / 100)
rides['fare'] = raw.clip(MIN_FARE, MAX_FARE)
rides['lyft_fee'] = np.minimum(FEE, rides['fare'])
rides['driver_pay'] = rides['fare'] - rides['lyft_fee']
print(rides[['miles', 'minutes', 'fare', 'driver_pay']].describe().round(2))
print('on the $5 floor', (rides.fare == MIN_FARE).sum())
print('hit $400 cap', (rides.fare == MAX_FARE).sum())


Median trip is short (~2.5 mi, ~12 min, ~$9). One ride hit the cap; left it in.


## One row per driver

Drop-off time for first/last trip. Some rides have no drop-off in the event log.


In [ ]:
dropoff = (
    ts.loc[ts.event == 'dropped_off_at', ['ride_id', 'timestamp']]
      .rename(columns={'timestamp': 'dropped_off_at'})
)
rt = rides.merge(dropoff, on='ride_id', how='left')
print('rides with no dropoff ts', rt.dropped_off_at.isna().sum())

per = rt.groupby('driver_id').agg(
    n_rides=('ride_id', 'count'),
    total_fare=('fare', 'sum'),
    total_driver_pay=('driver_pay', 'sum'),
    avg_fare=('fare', 'mean'),
    avg_miles=('miles', 'mean'),
    avg_minutes=('minutes', 'mean'),
    avg_prime=('ride_prime_time', 'mean'),
    first_trip=('dropped_off_at', 'min'),
    last_trip=('dropped_off_at', 'max'),
).reset_index()

df = drivers.merge(per, on='driver_id', how='left')
for c in ['n_rides', 'total_fare', 'total_driver_pay']:
    df[c] = df[c].fillna(0)

snapshot = ts.timestamp.max()
df['first_trip'] = pd.to_datetime(df['first_trip'])
df['last_trip'] = pd.to_datetime(df['last_trip'])

# +1 so same-day first/last still counts as a day
df['active_days'] = (df.last_trip - df.first_trip).dt.days + 1
df['active_days'] = df['active_days'].fillna(0)
df['days_since_onboard'] = (snapshot - df.driver_onboard_date).dt.days
df['days_since_last_trip'] = (snapshot - df.last_trip).dt.days
df['days_since_last_trip'] = df['days_since_last_trip'].fillna(df['days_since_onboard'])
df['rides_per_day'] = np.where(df.active_days > 0, df.n_rides / df.active_days, 0)
df['pay_per_day'] = np.where(df.active_days > 0, df.total_driver_pay / df.active_days, 0)

print(len(df), 'drivers,', int((df.n_rides == 0).sum()), 'with zero rides')
print(df[['n_rides', 'total_driver_pay', 'active_days', 'days_since_onboard', 'days_since_last_trip']].describe().round(2))


## Churn rule

Quiet 30 days before snapshot -> gone. Zero-ride drivers count as churned. Will mis-label some part-timers.


In [ ]:
GAP = 30
df['churned'] = df.days_since_last_trip > GAP

# left: onboard -> last trip. still around: onboard -> snapshot (observed, not finished)
df['lifetime_days'] = np.where(
    df.churned,
    (df.last_trip.fillna(df.driver_onboard_date) - df.driver_onboard_date).dt.days,
    (snapshot - df.driver_onboard_date).dt.days,
)
df.loc[df.n_rides == 0, 'lifetime_days'] = 0
df['lifetime_days'] = df['lifetime_days'].clip(lower=0)

print('churn rate {:.1%} ({}/{})'.format(df.churned.mean(), df.churned.sum(), len(df)))
print(df.groupby('churned').lifetime_days.describe().round(1))
print(df.groupby('churned')[['n_rides', 'total_driver_pay', 'active_days', 'rides_per_day']].mean().round(2))


## LTV

ltv_hist is what they already generated.

v1 used mean lifetime of people who already left (~13d). Actives have been onboarded 43-91 days so remaining days go to 0. Kept so the mistake is visible.

v2: actives get 30 more days at their own daily rate. Near-term, not a lifetime.


In [ ]:
df['ltv_hist'] = df.total_driver_pay
typ = df.loc[df.churned, 'lifetime_days'].mean()
df['remain_v1'] = np.where(~df.churned, np.maximum(typ - df.days_since_onboard, 0), 0)
df['ltv_v1'] = df.ltv_hist + df.remain_v1 * df.pay_per_day

EXTRA = 30
df['ltv_v2'] = df.ltv_hist + np.where(~df.churned, EXTRA * df.pay_per_day, 0)

print('v1 typical lifetime (churned mean) {:.1f}d'.format(typ))
print(df[['ltv_hist', 'ltv_v1', 'ltv_v2']].describe().round(2))
print(df.groupby('churned')[['ltv_hist', 'ltv_v1', 'ltv_v2', 'pay_per_day']].mean().round(2))
n = len(df)
print('total hist pay ${:,.0f}'.format(df.ltv_hist.sum()))
print('mean/median hist ${:,.0f} / ${:,.0f}'.format(df.ltv_hist.mean(), df.ltv_hist.median()))
print('mean/median v2 ${:,.0f} / ${:,.0f}'.format(df.ltv_v2.mean(), df.ltv_v2.median()))
print('top 20% share {:.1%}'.format(df.nlargest(int(0.2 * n), 'ltv_hist').ltv_hist.sum() / df.ltv_hist.sum()))


In [ ]:
started = df.n_rides > 0
print('churned life incl zeros {:.1f}d'.format(df.loc[df.churned, 'lifetime_days'].mean()))
print('churned life if they rode {:.1f}d n={}'.format(df.loc[df.churned & started, 'lifetime_days'].mean(), (df.churned & started).sum()))
print('actives days since onboard {:.1f}d'.format(df.loc[~df.churned, 'days_since_onboard'].mean()))


## Segments

Name them before clustering.


In [ ]:
def bucket(r):
    if r.n_rides == 0:
        return 'never_started'
    if r.churned:
        return 'early_exit'
    return 'workhorse'

df['segment'] = df.apply(bucket, axis=1)
seg = df.groupby('segment').agg(
    n=('driver_id', 'count'),
    rides=('n_rides', 'mean'),
    hist=('ltv_hist', 'mean'),
    v2=('ltv_v2', 'mean'),
    pay_day=('pay_per_day', 'mean'),
    life=('lifetime_days', 'mean'),
    prime=('avg_prime', 'mean'),
    miles=('avg_miles', 'mean'),
).round(1)
seg['share_drivers'] = (seg.n / seg.n.sum()).round(3)
pay = df.groupby('segment').ltv_hist.sum()
seg['share_pay'] = (pay / pay.sum()).round(3)
seg.sort_values('v2', ascending=False)


In [ ]:
order = ['never_started', 'early_exit', 'workhorse']
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(order, df.segment.value_counts().reindex(order).values)
ax.set_title('drivers by segment')
plt.show()
fig, ax = plt.subplots(figsize=(7, 4))
df.boxplot('ltv_hist', by='segment', ax=ax)
ax.set_title('historical driver-pay')
plt.suptitle('')
plt.show()


K-Means on three scaled columns as a check, not the definition.


In [ ]:
feat = df[['n_rides', 'pay_per_day', 'days_since_last_trip']].fillna(0)
X = StandardScaler().fit_transform(feat)
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['k3'] = km.fit_predict(X)
print(df.groupby('k3').agg(
    n=('driver_id', 'count'),
    rides=('n_rides', 'mean'),
    pay_day=('pay_per_day', 'mean'),
    recency=('days_since_last_trip', 'mean'),
    hist=('ltv_hist', 'mean'),
    pct_churned=('churned', 'mean'),
).round(1))
print(pd.crosstab(df.segment, df.k3, normalize='index').round(2))
# labels for this seed; remap if cluster ids swap
names = {0: 'high_workhorse', 1: 'steady_active', 2: 'exited'}
df['k3_name'] = df.k3.map(names)
print(df.k3_name.value_counts())


In [ ]:
df['lyft_fee_hist'] = df.n_rides * FEE
print('mean fee hist ${:,.0f}'.format(df.lyft_fee_hist.mean()))
print(df.groupby('k3_name').lyft_fee_hist.mean().round(0))


## Takeaways

- Hist LTV ~$2,006 mean / $1,890 median driver-pay. Near-term v2 ~$2,921 / $2,728.
- v1 collapsed because churned lifetime (~13d) is not the career of people still driving.
- 26% quiet at 30 days. Exits who rode lasted ~20 days. Actives: 68 days observed and still going.
- 74% workhorses hold 95% of pay. K-Means splits them into ~$66/day vs ~$21/day.
- Week-1 activation. Two live playbooks. Cheap win-back only for silent drivers.
- Plan a new onboard at $2.0k-$2.9k driver-pay, ~1 in 4 near zero. Do not annualize $66 x 365 off a 90-day file.
